# Limpieza de Grupo AD y Grupos BD

Este notebook limpia los archivos `Grupo AD.xlsx` y `Grupos BD.xlsx`, considerando las tabs `1` y `0` de ambos archivos.

El flujo incluye:
- carga de ambas hojas por archivo,
- uso de la primera fila como encabezado real,
- eliminacion de columnas vacias,
- normalizacion de nombres de columnas,
- completado de `cod_de_grupo` dentro de cada bloque,
- normalizacion de `cod_de_participante`, `nombre` y `telefono`,
- recodificacion de `edad`, `escolaridad` y `nivel_salarial` a categorias mas cortas,
- incorporacion de metadatos de condicion experimental,
- construccion de datasets de alta variedad, baja variedad y uno unificado.

## Alcance de la limpieza

Reglas principales que aplica este notebook:
- `cod_de_participante` se convierte a mayusculas.
- `nombre` se normaliza a formato titulo.
- `telefono` conserva solo numeros validos de 10 digitos; en otros casos queda como `NA`.
- `edad` se reduce a `Joven`, `Adulto` o `Mayor`.
- `escolaridad` se reduce a `Bachiller`, `Pregrado` o `Posgrado`.
- `nivel_salarial` se reduce a `Hasta 1 SMMLV`, `Entre 1 y 3 SMMLV` o `Mas de 3 SMMLV`.
- `grupo_experimental` resume la condicion de variedad y mision.

El notebook deja tres datasets finales:
- `dataset_alta_variedad`
- `dataset_baja_variedad`
- `dataset_unificado`

## 1. Imports

In [55]:
from pathlib import Path

import pandas as pd
import re
import unicodedata


## 2. Rutas de trabajo

In [56]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "Data"

grupo_ad_path = DATA_DIR / "Grupo AD.xlsx"
grupos_bd_path = DATA_DIR / "Grupos BD.xlsx"

grupo_ad_path, grupos_bd_path


(WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Grupo AD.xlsx'),
 WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Grupos BD.xlsx'))

## 3. Funciones auxiliares

In [57]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_header(value):
    value = normalize_text(value)
    value = unicodedata.normalize("NFKD", value)
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = value.lower().replace("°", "")
    value = re.sub(r"[^0-9a-z]+", "_", value)
    return value.strip("_")


def normalize_name(value):
    value = normalize_text(value)
    return value.title() if value else pd.NA


def clean_phone(value):
    value = normalize_text(value)
    if value == "":
        return pd.NA

    digits = re.sub(r"\D", "", value)
    match = re.search(r"\d{10}", digits)
    if match:
        return match.group(0)

    return pd.NA


def clean_age(value):
    value = normalize_text(value)
    if value == "":
        return pd.NA
    if "joven" in value.lower():
        return "Joven"
    if "adulto" in value.lower():
        return "Adulto"
    if "mayor" in value.lower():
        return "Mayor"
    return pd.NA


def clean_education(value):
    value = normalize_text(value)
    if value == "":
        return pd.NA
    lowered = value.lower()
    if "pregrado" in lowered:
        return "Pregrado"
    if "bachiller" in lowered:
        return "Bachiller"
    if "posgrado" in lowered or "postgrado" in lowered:
        return "Posgrado"
    return pd.NA


def clean_income_level(value):
    value = normalize_text(value)
    if value == "":
        return pd.NA
    lowered = value.lower()
    if "hasta 1" in lowered:
        return "Hasta 1 SMMLV"
    if "mas de 1 y menor de 3" in lowered or "más de 1 y menor de 3" in lowered:
        return "Entre 1 y 3 SMMLV"
    if "superior a 3" in lowered:
        return "Mas de 3 SMMLV"
    return pd.NA


def format_group_code(value):
    if pd.isna(value):
        return pd.NA
    value = normalize_text(value)
    digits = re.sub(r"\D", "", value)
    if digits == "":
        return value
    return digits.zfill(3)


def infer_condition(source_file, sheet_name):
    if source_file == "Grupo AD.xlsx":
        diversity_level = "alta"
        grupo_experimental = "alta_variedad_con_mision" if sheet_name == "1" else "alta_variedad_sin_mision"
    else:
        diversity_level = "baja"
        grupo_experimental = "baja_variedad_con_mision" if sheet_name == "1" else "baja_variedad_sin_mision"

    mission = 1 if sheet_name == "1" else 0
    return diversity_level, mission, grupo_experimental


def clean_sheet(workbook_path, sheet_name):
    raw = pd.read_excel(workbook_path, sheet_name=sheet_name)

    header_row = raw.iloc[0].tolist()
    valid_positions = [index for index, value in enumerate(header_row) if pd.notna(value)]
    raw = raw.iloc[:, : max(valid_positions) + 1]

    header = [normalize_header(value) for value in raw.iloc[0].tolist()]
    frame = raw.iloc[1:].copy()
    frame.columns = header
    frame = frame.dropna(how="all").reset_index(drop=True)

    if "cod_de_grupo" in frame.columns:
        frame["cod_de_grupo"] = pd.Series(frame["cod_de_grupo"], dtype="object").ffill()

    object_columns = frame.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        frame[column] = frame[column].apply(normalize_text)
        frame[column] = frame[column].replace("", pd.NA)

    if "cod_de_grupo" in frame.columns:
        frame["cod_de_grupo"] = frame["cod_de_grupo"].apply(format_group_code).astype("string")

    if "cod_de_participante" in frame.columns:
        frame["cod_de_participante"] = frame["cod_de_participante"].astype("string").str.strip().str.upper()

    if "nombre" in frame.columns:
        frame["nombre"] = frame["nombre"].apply(normalize_name).astype("string")

    if "correo" in frame.columns:
        frame["correo"] = frame["correo"].astype("string").str.lower()

    if "telefono" in frame.columns:
        frame["telefono"] = frame["telefono"].apply(clean_phone).astype("string")

    if "edad" in frame.columns:
        frame["edad"] = frame["edad"].apply(clean_age).astype("string")

    if "escolaridad" in frame.columns:
        frame["escolaridad"] = frame["escolaridad"].apply(clean_education).astype("string")

    if "nivel_salarial" in frame.columns:
        frame["nivel_salarial"] = frame["nivel_salarial"].apply(clean_income_level).astype("string")

    if "n" in frame.columns:
        frame = frame.drop(columns=["n"])

    diversity_level, mission, grupo_experimental = infer_condition(workbook_path.name, sheet_name)
    frame["source_file"] = workbook_path.name
    frame["source_sheet"] = sheet_name
    frame["diversity_level"] = diversity_level
    frame["mission"] = mission
    frame["grupo_experimental"] = grupo_experimental

    desired_order = [
        "cod_de_grupo",
        "cod_de_participante",
        "nombre",
        "genero",
        "raza",
        "edad",
        "correo",
        "telefono",
        "escolaridad",
        "nivel_salarial",
        "source_file",
        "source_sheet",
        "diversity_level",
        "mission",
        "grupo_experimental",
    ]
    present_columns = [column for column in desired_order if column in frame.columns]
    remaining_columns = [column for column in frame.columns if column not in present_columns]
    return frame[present_columns + remaining_columns]


def clean_workbook(workbook_path):
    cleaned_sheets = {}
    for sheet_name in ["1", "0"]:
        cleaned_sheets[sheet_name] = clean_sheet(workbook_path, sheet_name)

    combined = pd.concat(
        [cleaned_sheets["1"], cleaned_sheets["0"]],
        ignore_index=True,
        sort=False,
    )
    return cleaned_sheets, combined


## 4. Limpieza de Grupo AD

In [58]:
grupo_ad_sheets, grupo_ad_combined = clean_workbook(grupo_ad_path)

grupo_ad_sheets["1"].head()


,cod_de_grupo,cod_de_participante,nombre,genero,raza,edad,correo,telefono,escolaridad,nivel_salarial,source_file,source_sheet,diversity_level,mission,grupo_experimental
0,111,70G9NP,Geraldine Hernandez,Mujer,Mestizo,Adulto,ghdzm01@gmail.com,3147729217,Pregrado,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
1,111,06P068,Maria Fernanda Mogollon Viggiani,Mujer,Blanco,Joven,mafeviggi@gmail.com,3042086062,Pregrado,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
2,111,09701C,Said De Jesús Hernandez Ibarra,Hombre,Mestizo,Mayor,marggiehernandez1@gmail.com,<NA>,Bachiller,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
3,112,415MCF,Adalberto Escobar Castillo,Hombre,Mestizo,Adulto,aescobar2@cuc.edu.co,3014783827,Posgrado,Mas de 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
4,112,43OI0Q,Vanessa Vásquez Peñaloza,Mujer,Negro o afrodescendiente,Joven,vanessavasquez1999@hotmail.com,3043804281,Pregrado,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision


In [59]:
grupo_ad_sheets["0"].head()


,cod_de_grupo,cod_de_participante,nombre,genero,raza,edad,correo,telefono,escolaridad,nivel_salarial,source_file,source_sheet,diversity_level,mission,grupo_experimental
0,101,P096ER,Lina Zambrano,Mujer,Blanco,Joven,mhormech1@cuc.edu.co,3046706817,Pregrado,Hasta 1 SMMLV,Grupo AD.xlsx,0,alta,0,alta_variedad_sin_mision
1,101,3L22H0,Said Castro Comas,Hombre,Mestizo,Adulto,<NA>,<NA>,Bachiller,Hasta 1 SMMLV,Grupo AD.xlsx,0,alta,0,alta_variedad_sin_mision
2,101,7533F8,Osvaldo Arevalo,Hombre,Indígena,Mayor,osvaldoarevalo220668@gmail.com,3183788759,Posgrado,Mas de 3 SMMLV,Grupo AD.xlsx,0,alta,0,alta_variedad_sin_mision
3,102,446KZU,Miangely Peñaloza,Mujer,Mestizo,Joven,mpesar0212@gmail.com,<NA>,Pregrado,Hasta 1 SMMLV,Grupo AD.xlsx,0,alta,0,alta_variedad_sin_mision
4,102,9TV059,Jorge Borda,Hombre,Mestizo,Mayor,jorgeborda30@hotmail.com,3107021547,Posgrado,Mas de 3 SMMLV,Grupo AD.xlsx,0,alta,0,alta_variedad_sin_mision


In [60]:
grupo_ad_combined.shape


(120, 15)

## 5. Limpieza de Grupos BD

In [61]:
grupos_bd_sheets, grupos_bd_combined = clean_workbook(grupos_bd_path)

grupos_bd_sheets["1"].head()


,cod_de_grupo,cod_de_participante,nombre,genero,raza,edad,correo,telefono,escolaridad,nivel_salarial,source_file,source_sheet,diversity_level,mission,grupo_experimental
0,011,MV8313,Sara Camila Ruiz Vega,Mujer,Blanco,Joven,camilaruiz092019@gmail.com,3212440849,Pregrado,Hasta 1 SMMLV,Grupos BD.xlsx,1,baja,1,baja_variedad_con_mision
1,011,63YFUS,Valentina Salcedo,Mujer,Blanco,Joven,valentinasalcedo49@gmail.com,3023417517,Pregrado,Entre 1 y 3 SMMLV,Grupos BD.xlsx,1,baja,1,baja_variedad_con_mision
2,011,HVD803,Stefany Gutierrez Retamozo,Mujer,Blanco,Adulto,stefanypgr2@gmail.com,<NA>,Posgrado,Entre 1 y 3 SMMLV,Grupos BD.xlsx,1,baja,1,baja_variedad_con_mision
3,012,Q4368H,Sailly Mailin Arias Vasquuez,Mujer,Mestizo,Joven,samarva.2307@gmail.com,<NA>,Pregrado,Entre 1 y 3 SMMLV,Grupos BD.xlsx,1,baja,1,baja_variedad_con_mision
4,012,0QQ81N,Luis Enrique Fernandez Arteta,Hombre,Blanco,Joven,lfernand36@cuc.edu.co,<NA>,Pregrado,Hasta 1 SMMLV,Grupos BD.xlsx,1,baja,1,baja_variedad_con_mision


In [62]:
grupos_bd_sheets["0"].head()


,cod_de_grupo,cod_de_participante,nombre,genero,raza,edad,correo,telefono,escolaridad,nivel_salarial,source_file,source_sheet,diversity_level,mission,grupo_experimental
0,001,9D2V65,Yurizan Barrios Vides,Mujer,Mestizo,Adulto,yubarvi@gmail.com,3043881360,Posgrado,Entre 1 y 3 SMMLV,Grupos BD.xlsx,0,baja,0,baja_variedad_sin_mision
1,001,9X7HD6,David Barrios,Hombre,Mestizo,Adulto,dbarrios21@cuc.edu.co,5731625371,Bachiller,Hasta 1 SMMLV,Grupos BD.xlsx,0,baja,0,baja_variedad_sin_mision
2,001,VR9614,Rosa Margarita Benítez Domínguez,Mujer,Mestizo,Adulto,margarita_0211@hotmail.com,<NA>,Pregrado,Hasta 1 SMMLV,Grupos BD.xlsx,0,baja,0,baja_variedad_sin_mision
3,002,YN36DA,Carla Marcela Chavez Barraza,Mujer,Blanco,Joven,carlachavezbarraza49@gamil.com,3002329303,Pregrado,Hasta 1 SMMLV,Grupos BD.xlsx,0,baja,0,baja_variedad_sin_mision
4,002,YL10BD,Nayelis Esther Hernandez Villegas,Mujer,Mestizo,Joven,nhernandezv@cuc.edu.co,3043336636,Pregrado,Hasta 1 SMMLV,Grupos BD.xlsx,0,baja,0,baja_variedad_sin_mision


In [63]:
grupos_bd_combined.shape


(120, 15)

## 6. Datasets finales

In [64]:
dataset_alta_variedad = grupo_ad_combined.copy()
dataset_baja_variedad = grupos_bd_combined.copy()

dataset_unificado = pd.concat(
    [dataset_alta_variedad, dataset_baja_variedad],
    ignore_index=True,
    sort=False,
)

dataset_unificado.head()


,cod_de_grupo,cod_de_participante,nombre,genero,raza,edad,correo,telefono,escolaridad,nivel_salarial,source_file,source_sheet,diversity_level,mission,grupo_experimental
0,111,70G9NP,Geraldine Hernandez,Mujer,Mestizo,Adulto,ghdzm01@gmail.com,3147729217,Pregrado,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
1,111,06P068,Maria Fernanda Mogollon Viggiani,Mujer,Blanco,Joven,mafeviggi@gmail.com,3042086062,Pregrado,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
2,111,09701C,Said De Jesús Hernandez Ibarra,Hombre,Mestizo,Mayor,marggiehernandez1@gmail.com,<NA>,Bachiller,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
3,112,415MCF,Adalberto Escobar Castillo,Hombre,Mestizo,Adulto,aescobar2@cuc.edu.co,3014783827,Posgrado,Mas de 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision
4,112,43OI0Q,Vanessa Vásquez Peñaloza,Mujer,Negro o afrodescendiente,Joven,vanessavasquez1999@hotmail.com,3043804281,Pregrado,Entre 1 y 3 SMMLV,Grupo AD.xlsx,1,alta,1,alta_variedad_con_mision


In [65]:
dataset_alta_variedad.shape, dataset_baja_variedad.shape, dataset_unificado.shape


((120, 15), (120, 15), (240, 15))

## 7. Validaciones rapidas

In [66]:
dataset_unificado.isna().sum().sort_values(ascending=False)


telefono               126
correo                   5
cod_de_participante      0
nombre                   0
genero                   0
raza                     0
cod_de_grupo             0
edad                     0
escolaridad              0
nivel_salarial           0
source_file              0
source_sheet             0
diversity_level          0
mission                  0
grupo_experimental       0
dtype: int64

In [67]:
dataset_unificado.groupby(["diversity_level", "mission", "grupo_experimental"]).size()


diversity_level  mission  grupo_experimental      
alta             0        alta_variedad_sin_mision    60
                 1        alta_variedad_con_mision    60
baja             0        baja_variedad_sin_mision    60
                 1        baja_variedad_con_mision    60
dtype: int64

## 8. Exportacion a Excel

Esta celda guarda los tres datasets limpios en la carpeta `Data` de la raiz del repositorio.

In [68]:
output_dir = DATA_DIR

alta_path = output_dir / "Grupo_AD_clean.xlsx"
baja_path = output_dir / "Grupos_BD_clean.xlsx"
unificado_path = output_dir / "Grupos_AD_BD_unificado.xlsx"

dataset_alta_variedad.to_excel(alta_path, index=False)
dataset_baja_variedad.to_excel(baja_path, index=False)
dataset_unificado.to_excel(unificado_path, index=False)

alta_path, baja_path, unificado_path


(WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Grupo_AD_clean.xlsx'),
 WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Grupos_BD_clean.xlsx'),
 WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Grupos_AD_BD_unificado.xlsx'))